In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [3]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [15]:
instrument_id = '511520'
trade_ymd = '20260319'
day = 5

In [68]:
param_dict = {
    'name' : 'bollingger_bands',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,

    'ma_window': 300,
    'std_window': 300,
    'beta': 3
}

In [69]:
%%time
import os
import numpy as np
from collections import deque
import math

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        
        self.__dict__.update(param_dict)
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl' 
        if os.path.exists(data_file):
            os.remove(data_file)

        self.position_last = 0

        self.max_len = max(self.ma_window,self.std_window)
        self.price_list = deque(maxlen=self.max_len)

        self.prev_signal = 0
        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return
        
        self.price_list.append(price)

        if(len(self.price_list) < self.max_len):
            self.position_last = 0
            self.prev_signal = 0
            return

        window_data = list(self.price_list)[-self.max_len:]

        ma_slice = window_data[-self.ma_window:]
        ma = sum(ma_slice) / self.ma_window

        std_slice = window_data[-self.std_window:]
        mean_of_squares = sum(x * x for x in std_slice) / self.std_window
        mean = sum(std_slice) / self.std_window
        square_of_mean = mean * mean
        variance = max(0, mean_of_squares - square_of_mean)
        std = math.sqrt(variance)

        if std > ma * 0.5 or std < 0.000001:
            print(f"⚠️ 异常波动警告: price={price}, ma={ma}, std={std}")
            return
        
        upper_band = ma + self.beta * std
        lower_band = ma - self.beta * std

        signal = self.prev_signal 

        if self.position_last == 0:
            if price < lower_band:
                signal = 1  
            elif price > upper_band:
                signal = -1
            
        elif self.position_last == 1: # 持有多单
            # 平仓条件：价格回归到均线之上，或者价格触及上轨（反向信号）
            if price >= ma or price >= upper_band:
                signal = 0 
            # 否则保持持有 (signal = 1)

        elif self.position_last == -1: # 持有空单
            # 平仓条件：价格回归到均线之下，或者价格触及下轨（反向信号）
            if price <= ma or price <= lower_band:
                signal = 0
            # 否则保持持有 (signal = -1)

        # 5. 更新状态
        self.position_last = signal
        self.prev_signal = signal
        
        return

CPU times: user 17 μs, sys: 0 ns, total: 17 μs
Wall time: 18.8 μs


In [70]:
%%time
def backtest_demo(instrument_id, trade_ymd, strategy_name):
    strategy = StrategyDemo(param_dict)
    snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
    position_dict = {}
    for snap in snap_list[:]:
        strategy.on_snap(snap)
        position_dict[snap['time_mark']] = strategy.position_last
    profit = base_tool.backtest_quick(instrument_id, trade_ymd, strategy_name, position_dict)
    return profit
backtest_demo(instrument_id, trade_ymd, param_dict['name'])

/home/jovyan/work/backtest_result/511520_20260319_bollingger_bands.pkl Please wait
CPU times: user 245 ms, sys: 14.8 ms, total: 260 ms
Wall time: 560 ms


,order_time,order_price,total,trade,cancel,hold,profit_last,profits,maxdd,MAR,pper
0,2026-03-19 09:52:40,116.078,1,1,0,-100,0.0,0.0,1.0,0.00,0.00
1,2026-03-19 10:01:55,116.106,2,2,0,0,-2.8,-2.8,2.8,-1.00,-1.40
2,2026-03-19 10:07:25,116.103,3,3,0,-100,0.0,-2.8,2.8,-1.00,-0.93
3,2026-03-19 10:17:18,116.143,4,4,0,0,-4.0,-6.8,6.8,-1.00,-1.70
4,2026-03-19 10:25:59,116.125,5,5,0,100,0.0,-6.8,6.8,-1.00,-1.36
...,...,...,...,...,...,...,...,...,...,...,...
53,2026-03-19 14:39:59,116.187,56,54,2,0,-0.5,-16.2,16.2,-1.00,-0.30
54,2026-03-19 14:46:05,116.185,57,55,2,-100,0.0,-16.2,16.2,-1.00,-0.29
55,2026-03-19 14:46:59,116.185,58,56,2,0,0.1,-16.1,16.2,-0.99,-0.29
56,2026-03-19 14:47:35,116.180,59,57,2,100,0.0,-16.1,16.2,-0.99,-0.28
